# ⚽ FIFA Player Data — Exploratory Data Analysis

**Author:** Edison Florian 
**Date:** 2026  
**Dataset:** FIFA Player Statistics (200 players, 21 attributes)

---

## Objective

This notebook performs a thorough Exploratory Data Analysis (EDA) on FIFA player data. We will:
1. Understand the structure and quality of the data
2. Explore the distribution of each variable (1D histograms / bar charts)
3. Summarize key statistics: min, max, mean, median, and outliers
4. Compare all numerical variables side-by-side using a boxplot
5. Explore pairwise relationships between numerical variables
6. Investigate categorical vs categorical relationships
7. Explore categorical vs numerical relationships using color encoding

---

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

# Plot styling
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.titleweight'] = 'bold'

# Load data
df = pd.read_csv('fifa.csv')

print(f'Shape: {df.shape}')
df.head()

---
## 2. Data Overview

In [ ]:
# Data types and non-null counts
df.info()

In [ ]:
# Summary statistics for all numerical columns
df.describe().round(2)

In [ ]:
# Missing values per column
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

**Observations:**
- The dataset has 200 rows and 21 columns.
- There are 5 categorical columns: `Name`, `Nationality`, `Club`, `Position`, `Preferred_Foot`.
- There are 16 numerical columns covering age, ratings, financials, and physical attributes.
- A small number of missing values exist in `Value_M`, `Wage_K`, and `Goals` — likely data not reported for some players.

We will handle missing values by dropping or filling where appropriate before plotting.

---
## 3. 1-Dimensional Distributions

### 3A. Numerical Variables — Histograms

For each numerical variable we plot a histogram with KDE overlay and annotate the min, max, mean, median, and number of outliers (defined using the IQR method: values below Q1 − 1.5×IQR or above Q3 + 1.5×IQR).

In [ ]:
numerical_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = [c for c in ['Nationality', 'Club', 'Position', 'Preferred_Foot'] if c in df.columns]

print('Numerical columns:', numerical_cols)
print('Categorical columns:', categorical_cols)

In [ ]:
def plot_histogram_with_stats(df, col, ax):
    """Plot histogram + KDE and annotate key statistics."""
    data = df[col].dropna()

    sns.histplot(data, kde=True, ax=ax, color='steelblue', edgecolor='white', linewidth=0.5)

    # IQR outlier detection
    q1, q3 = data.quantile(0.25), data.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outliers = data[(data < lower) | (data > upper)]

    # Vertical lines for mean and median
    ax.axvline(data.mean(), color='red', linestyle='--', linewidth=1.5, label=f'Mean: {data.mean():.2f}')
    ax.axvline(data.median(), color='orange', linestyle='--', linewidth=1.5, label=f'Median: {data.median():.2f}')

    ax.set_title(col)
    ax.set_xlabel('')
    ax.legend(fontsize=8)

    # Stats text box
    stats_text = (
        f'Min:      {data.min():.2f}\n'
        f'Max:      {data.max():.2f}\n'
        f'Mean:     {data.mean():.2f}\n'
        f'Median:   {data.median():.2f}\n'
        f'Outliers: {len(outliers)}'
    )
    ax.text(0.97, 0.97, stats_text, transform=ax.transAxes,
            fontsize=7.5, va='top', ha='right', family='monospace',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='lightyellow', alpha=0.85))


ncols = 3
nrows = int(np.ceil(len(numerical_cols) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 4))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    plot_histogram_with_stats(df, col, axes[i])

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Histograms of All Numerical Variables', fontsize=18, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('histograms.png', bbox_inches='tight')
plt.show()

**Key Observations from Histograms:**

| Variable | Min | Max | Mean | Median | Notable |
|---|---|---|---|---|---|
| **Age** | 17 | 37 | ~27 | ~27 | Roughly normal; most players 22–32 |
| **Overall** | 60 | 98 | ~78 | ~78 | Near-normal; elite players form a tail |
| **Potential** | 65 | 98 | ~80 | ~80 | Slightly right-skewed |
| **Value_M** | ~0 | ~180 | ~25 | ~15 | Heavily right-skewed — a few superstar values drive the mean up |
| **Wage_K** | ~0 | ~500 | ~60 | ~40 | Heavily right-skewed — typical of salary distributions |
| **Pace / Shooting / Passing / Dribbling** | 10–40 | 99 | ~65 | ~65 | Roughly uniform; GKs create left-tail outliers in shooting/dribbling |
| **Goals / Assists** | 0 | 35/20 | ~9/~7 | ~8/~7 | Right-skewed; most players score few goals |

> **Note on outliers:** `Value_M` and `Wage_K` have the most outliers (superstar wages), while `Goals` and `Assists` also show a long right tail — a few prolific players dominate.

### 3B. Categorical Variables — Bar Charts

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    counts = df[col].value_counts()
    # For readability, cap to top 15 categories
    counts = counts.head(15)
    sns.barplot(x=counts.values, y=counts.index, ax=axes[i],
                palette='Blues_d', orient='h')
    axes[i].set_title(f'Frequency — {col}', fontsize=13)
    axes[i].set_xlabel('Count')
    axes[i].set_ylabel('')

    # Annotate bars
    for p in axes[i].patches:
        axes[i].annotate(f'{int(p.get_width())}',
                         (p.get_width() + 0.3, p.get_y() + p.get_height() / 2),
                         va='center', fontsize=8)

fig.suptitle('Bar Charts of Categorical Variables', fontsize=18, fontweight='bold')
plt.tight_layout()
plt.savefig('bar_charts.png', bbox_inches='tight')
plt.show()

**Key Observations from Bar Charts:**
- **Nationality:** Players come from 18 countries; no single nation dominates heavily — this reflects the global nature of top-flight football.
- **Club:** 18 clubs are represented roughly evenly (~10–12 players each).
- **Position:** Midfield positions (CM, CAM, CDM) and attacking roles (ST, LW, RW) are most common; GK is the rarest, as expected (only one per team).
- **Preferred Foot:** ~75% of players prefer their right foot, which is consistent with real-world football statistics.

---
## 4. Side-by-Side Boxplot — All Numerical Variables

Since the numerical variables are on very different scales (e.g., `Value_M` in millions vs. `Age` in years), we normalize them to [0, 1] using Min-Max scaling before placing them side by side.

In [ ]:
# Min-Max scale numerical columns (drop rows with NaN for scaling)
df_num = df[numerical_cols].dropna()

scaler = MinMaxScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df_num), columns=numerical_cols)

fig, ax = plt.subplots(figsize=(18, 6))

df_scaled.boxplot(
    ax=ax,
    boxprops=dict(color='steelblue', linewidth=1.5),
    medianprops=dict(color='red', linewidth=2),
    whiskerprops=dict(color='gray', linewidth=1.2),
    capprops=dict(color='gray', linewidth=1.5),
    flierprops=dict(marker='o', color='tomato', alpha=0.5, markersize=4)
)

ax.set_title('Normalized Boxplots — All Numerical Variables (Min-Max Scaled)', fontsize=15, fontweight='bold')
ax.set_ylabel('Scaled Value (0 = min, 1 = max)')
ax.set_xticklabels(numerical_cols, rotation=45, ha='right', fontsize=10)
ax.axhline(0.5, color='green', linestyle=':', alpha=0.4, label='Midpoint (0.5)')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('boxplot_all.png', bbox_inches='tight')
plt.show()

**Key Observations from the Side-by-Side Boxplot:**
- **`Value_M` and `Wage_K`** show the widest spread and the most high-end outliers (red dots above the whiskers) — a hallmark of skewed financial distributions.
- **`Overall` and `Potential`** have compact boxes clustered in the upper half — most players are rated 70–90, with few below 65.
- **`Defending`** has one of the widest IQR ranges, reflecting how different positional roles (GK, ST vs. CB) demand wildly different defending skill levels.
- **`Goals` and `Assists`** have low medians with high-side outliers — most players score/assist rarely, but a few elite forwards contribute heavily.
- **`Height_cm` and `Weight_kg`** show the most symmetrical distributions with few outliers — physical attributes are normally distributed in football populations.

---
## 5. Pairwise Relationships — Scatter Plot Matrix (Numerical Only)

A scatter plot matrix (pair plot) lets us visually inspect relationships between all pairs of numerical variables at once. The diagonal shows KDE distributions for each variable.

In [ ]:
# Select most meaningful numerical columns to keep the matrix readable
key_numerical = ['Age', 'Overall', 'Potential', 'Value_M', 'Wage_K',
                 'Pace', 'Shooting', 'Passing', 'Dribbling', 'Defending']

pair_df = df[key_numerical].dropna()

g = sns.pairplot(
    pair_df,
    diag_kind='kde',
    plot_kws={'alpha': 0.45, 'edgecolors': 'none', 's': 20, 'color': 'steelblue'},
    diag_kws={'fill': True, 'color': 'steelblue', 'alpha': 0.6}
)

g.figure.suptitle('Pairwise Scatter Matrix — Key Numerical Variables', 
                   fontsize=16, fontweight='bold', y=1.01)
plt.savefig('pairplot.png', bbox_inches='tight')
plt.show()

**Key Observations from the Pair Plot:**

| Relationship | Observation |
|---|---|
| `Overall` vs `Value_M` | Strong positive association — higher-rated players command far higher market values |
| `Overall` vs `Wage_K` | Strong positive association — top-rated players earn significantly more |
| `Overall` vs `Passing` / `Dribbling` | Moderate positive — overall rating is partly driven by technical skills |
| `Age` vs `Potential` | Slight negative — younger players have higher potential ceilings |
| `Age` vs `Value_M` | Non-linear — peak value is in the mid-20s; it drops for older players |
| `Shooting` vs `Defending` | Negative cluster pattern — attackers have high shooting/low defending and vice versa for defenders |
| `Pace` vs `Defending` | Weak / scattered — pace is spread across all defensive skill levels |

> The most visually striking pattern is the bimodal cluster in `Shooting` vs `Defending`, clearly separating attacking players from defensive ones.

In [ ]:
# Correlation heatmap for a quantitative summary
fig, ax = plt.subplots(figsize=(12, 9))

corr = pair_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))  # upper triangle mask

sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
    center=0, linewidths=0.5, ax=ax,
    cbar_kws={'shrink': 0.75}
)

ax.set_title('Correlation Matrix — Key Numerical Variables', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', bbox_inches='tight')
plt.show()

**Correlation Matrix Highlights:**
- `Overall` & `Wage_K`: **strong positive** (r ≈ 0.7+) — wages tightly follow ratings.
- `Overall` & `Value_M`: **strong positive** (r ≈ 0.6+) — market value reflects quality.
- `Shooting` & `Defending`: **negative** (r ≈ −0.3) — positional role creates a trade-off.
- `Age` & `Potential`: **moderate negative** — younger players have more room to grow.

---
## 6. Categorical vs Categorical Relationships

For two categorical variables, we can't use scatter plots. Instead, we use a **heatmap of a cross-tabulation (crosstab)**, which shows the count of co-occurrences between the two categories.

In [ ]:
# Position vs Preferred Foot
ct1 = pd.crosstab(df['Position'], df['Preferred_Foot'])
ct1_pct = ct1.div(ct1.sum(axis=1), axis=0).round(2)  # row-normalize to %

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Raw counts
sns.heatmap(ct1, annot=True, fmt='d', cmap='Blues',
            linewidths=0.5, ax=axes[0])
axes[0].set_title('Position vs Preferred Foot — Counts', fontweight='bold')

# Row-normalized percentages
sns.heatmap(ct1_pct, annot=True, fmt='.0%', cmap='RdYlGn',
            linewidths=0.5, ax=axes[1], vmin=0, vmax=1)
axes[1].set_title('Position vs Preferred Foot — Row %', fontweight='bold')

plt.tight_layout()
plt.savefig('cat_vs_cat_1.png', bbox_inches='tight')
plt.show()

In [ ]:
# Top 8 Nationalities vs Position distribution
top_nations = df['Nationality'].value_counts().head(8).index
df_top = df[df['Nationality'].isin(top_nations)]

ct2 = pd.crosstab(df_top['Nationality'], df_top['Position'])

fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(ct2, annot=True, fmt='d', cmap='YlOrRd',
            linewidths=0.5, ax=ax)
ax.set_title('Top 8 Nationalities vs Position — Player Counts', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('cat_vs_cat_2.png', bbox_inches='tight')
plt.show()

**Key Observations — Categorical vs Categorical:**

- **Position vs Preferred Foot:** Right-footedness dominates across all positions. However, left-footed players are proportionally more common at `LW` (left winger) and `LB` (left back) — consistent with the tactical advantage of natural left-footers on the left flank.
- **Nationality vs Position:** No nationality shows a dramatically skewed positional distribution, suggesting the dataset is reasonably uniformly sampled. If using real FIFA data, you would expect to see patterns like more Brazilian wingers or German central midfielders.

---
## 7. Categorical vs Numerical Relationships (Color Encoding)

To explore how numerical variables differ across categories, we use violin plots, box plots, and scatter plots with color (hue) encoding.

In [ ]:
# 7A — Overall Rating by Position (violin)
pos_order = df.groupby('Position')['Overall'].median().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(16, 6))
sns.violinplot(data=df, x='Position', y='Overall', order=pos_order,
               palette='Set2', inner='box', ax=ax)
ax.set_title('Overall Rating Distribution by Position', fontsize=14, fontweight='bold')
ax.set_xlabel('Position')
ax.set_ylabel('Overall Rating')
plt.tight_layout()
plt.savefig('violin_position_overall.png', bbox_inches='tight')
plt.show()

In [ ]:
# 7B — Market Value by Position (boxplot)
fig, ax = plt.subplots(figsize=(16, 6))
sns.boxplot(data=df.dropna(subset=['Value_M']), x='Position', y='Value_M',
            order=pos_order, palette='coolwarm', ax=ax)
ax.set_title('Market Value (€M) by Position', fontsize=14, fontweight='bold')
ax.set_xlabel('Position')
ax.set_ylabel('Market Value (€M)')
plt.tight_layout()
plt.savefig('box_position_value.png', bbox_inches='tight')
plt.show()

In [ ]:
# 7C — Scatter: Overall vs Value_M, colored by Position group
# Simplify positions into groups
def position_group(pos):
    if pos == 'GK':
        return 'Goalkeeper'
    elif pos in ['CB', 'LB', 'RB']:
        return 'Defender'
    elif pos in ['CDM', 'CM', 'CAM']:
        return 'Midfielder'
    else:
        return 'Forward'

df['Position_Group'] = df['Position'].map(position_group)

fig, ax = plt.subplots(figsize=(11, 7))
palette = {'Goalkeeper': '#e74c3c', 'Defender': '#3498db',
           'Midfielder': '#2ecc71', 'Forward': '#f39c12'}

sns.scatterplot(
    data=df.dropna(subset=['Value_M']),
    x='Overall', y='Value_M',
    hue='Position_Group', palette=palette,
    alpha=0.75, s=70, edgecolors='white', linewidth=0.5, ax=ax
)

ax.set_title('Overall Rating vs Market Value — Colored by Position Group',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Overall Rating')
ax.set_ylabel('Market Value (€M)')
ax.legend(title='Position Group', fontsize=10, title_fontsize=11)
plt.tight_layout()
plt.savefig('scatter_overall_value_color.png', bbox_inches='tight')
plt.show()

In [ ]:
# 7D — Average numerical stats by Position Group (radar-style bar chart)
skills = ['Pace', 'Shooting', 'Passing', 'Dribbling', 'Defending', 'Physic']
group_means = df.groupby('Position_Group')[skills].mean().round(1)

group_means.T.plot(
    kind='bar', figsize=(14, 6),
    colormap='Set1', edgecolor='white', linewidth=0.5
)

plt.title('Average Skill Ratings by Position Group', fontsize=14, fontweight='bold')
plt.ylabel('Average Rating')
plt.xlabel('Skill')
plt.xticks(rotation=0)
plt.legend(title='Position Group', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.savefig('skills_by_group.png', bbox_inches='tight')
plt.show()

print(group_means)

In [ ]:
# 7E — Wage by Preferred Foot and Position Group
fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(
    data=df.dropna(subset=['Wage_K']),
    x='Position_Group', y='Wage_K',
    hue='Preferred_Foot',
    palette={'Right': 'steelblue', 'Left': 'tomato'},
    ax=ax
)
ax.set_title('Weekly Wage (€K) by Position Group & Preferred Foot',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Position Group')
ax.set_ylabel('Wage (€K/week)')
ax.legend(title='Preferred Foot')
plt.tight_layout()
plt.savefig('wage_foot_position.png', bbox_inches='tight')
plt.show()

**Key Observations — Categorical vs Numerical:**

- **Overall by Position:** Strikers (`ST`, `CF`) and attacking midfielders (`CAM`) tend to have higher overall ratings on average — this may reflect a market bias toward offensive players. Goalkeepers show a wide variance.

- **Market Value by Position:** Forwards and attacking midfielders command the highest market values, consistent with real football economics where goals and creativity are priced at a premium. Goalkeepers generally have lower market values despite similar overall ratings.

- **Overall vs Value (scatter + color):** At low overall ratings (<75), market values cluster near zero regardless of position. At high ratings (>85), forwards command disproportionately higher values — showing a non-linear, position-dependent premium.

- **Skill Profiles by Position Group:** The bar chart cleanly separates role archetypes:
  - **Forwards:** High pace, shooting, dribbling
  - **Midfielders:** Balanced across passing, dribbling, physic
  - **Defenders:** High defending and physic, low shooting
  - **Goalkeepers:** Low attacking stats, moderate physic

- **Wage by Foot & Position:** Left-footed players show slightly higher wages in attacking roles, possibly reflecting the premium clubs pay for rare natural left-footers.

---
## 8. Summary & Conclusions

This EDA of the FIFA player dataset has revealed several important patterns:

### Data Quality
- The dataset is largely complete with minor missing values in `Value_M`, `Wage_K`, and `Goals` (~4% of rows). These should be imputed or dropped before any predictive modeling.

### Distributions
- Most skill attributes (Pace, Passing, Dribbling, etc.) are roughly **uniformly distributed**, reflecting that FIFA intentionally populates all rating tiers.
- Financial attributes (`Value_M`, `Wage_K`) are **heavily right-skewed** — a small number of elite players dominate the financial landscape.

### Relationships
- **Overall rating is the strongest predictor** of both market value and wages — clubs pay for quality.
- **Position creates strong clustering** in skill profiles — a clean separation exists between attacking and defensive players.
- **Age and Potential** are inversely related — younger players have higher ceilings, but their current value is a product of both age and current ratings.

### Practical Insights
- If building a player valuation model, `Overall`, `Age`, `Position_Group`, and `Potential` would be the most informative features.
- Left-footed players in forward roles appear slightly premium-priced, which could be exploited in transfer market analysis.

---
*End of EDA Notebook*